# P8 差は本物か

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

区間・0 件の上限・対の比較（McNemar・ブートストラップ・並べ替え）・塊の標準誤差・必要な件数・多重比較を、講師の本物の実行の結果（lab03 と lab09 の out/）で計算します。LLM は呼びません。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 1. 同じ試験をもう一度やったら（模擬）
本当の正答率が 90% のモデルでも、問いを引き直すたびに点は動きます。

In [ ]:
import random
from common import kit, stats
rng = random.Random(1)
accs = []
for t in range(10):
    k = sum(rng.random() < 0.9 for _ in range(120))
    lo, hi = stats.wilson(k, 120)
    accs.append(k / 120)
    print(f"{t + 1:2d} 回目: {k}/120 = {kit.pct(k / 120)}  95%区間 {kit.ci_str(lo, hi)}"
          f"  90% を {'含む' if lo <= 0.9 <= hi else '外す'}")
print(f"最小 {kit.pct(min(accs))} 〜 最大 {kit.pct(max(accs))}（同じモデル・同じ腕前でも、これだけ動く）")

## 2. Wilson の区間
`common/stats.py` の本物のコードと、正規近似との違い。

In [ ]:
show(stats.wilson)
rows = []
for k, n, what in [(120, 120, "分類 v1（両モデル）"), (48, 48, "分類 v2 難問"), (108, 120, "仮に 108/120"),
                   (0, 12, "漏えい（lab11）")]:
    lo, hi = stats.wilson(k, n)
    p = k / n
    se = (p * (1 - p) / n) ** 0.5
    rows.append([what, f"{k}/{n}", kit.ci_str(lo, hi), kit.ci_str(max(0, p - 1.96 * se), min(1, p + 1.96 * se))])
print()
print(kit.table(rows, ["実験", "結果", "Wilson 95%", "正規近似 95%"]))

## 3. 0 件でも安全とは言えない ― 3 の法則
0/n のときの上限（95%）は、だいたい 3/n。

In [ ]:
rows = []
for n in (12, 30, 120, 300, 1500):
    lo, hi = stats.wilson(0, n)
    rows.append([n, f"0/{n}", kit.pct(hi), kit.pct(stats.rule_of_three(n))])
print(kit.table(rows, ["件数", "結果", "Wilson の上限", "3 の法則 3/n"]))
print("上限を 0.2% まで下げたいなら 3 / 0.002 =", round(3 / 0.002), "件（それで 0 件なら）")

## 4. 対の比較 ― 食い違った問いだけを見る
基準（全文の手引き）と候補（短くした依頼文）を、同じ v2 難問 48 問で比べます（Sonnet・本物の実行の記録）。

In [ ]:
base = {r["id"]: r for r in kit.read_jsonl("lab03_classify_extract/out/runs_classify_v2_sonnet-5-5.jsonl")}
cand = {r["id"]: r for r in kit.read_jsonl("lab09_ci_gate/out/runs_candidate_v2_sonnet-5-5.jsonl")}
ids = sorted(base)
a, b = [base[i]["correct"] for i in ids], [cand[i]["correct"] for i in ids]
both = sum(x and y for x, y in zip(a, b))
none = sum((not x) and (not y) for x, y in zip(a, b))
only_a, only_b = stats.discordant(a, b)
print(kit.table([["基準 ○", both, only_a], ["基準 ×", only_b, none]], ["", "候補 ○", "候補 ×"]))
print(f"\n両方正解 {both} 問・両方不正解 {none} 問は、差について何も言わない")
print(f"見るのは食い違った {only_a + only_b} 問だけ: 基準だけ正解 {only_a} / 候補だけ正解 {only_b}")

## 5. 対のブートストラップと並べ替え検定
同じ問いの差（候補 − 基準）を引き直して、差の区間と p 値を出します。

In [ ]:
show(stats.paired_bootstrap)
pb = stats.paired_bootstrap(a, b, seed=0)
p_mc = stats.mcnemar_exact(only_a, only_b)
p_perm = stats.paired_permutation(a, b, B=5000, seed=0)
print(f"\n正答 {sum(a)} → {sum(b)}  差 {pb['diff']:+.3f}  対のブートストラップ 95%区間 [{pb['lo']:+.3f}, {pb['hi']:+.3f}]")
print(f"McNemar p={p_mc:.2g} / 並べ替え p={p_perm:.2g}")
la, ha = stats.wilson(sum(a), len(a))
lb, hb = stats.wilson(sum(b), len(b))
print(f"対にしないで区間だけ並べると: 基準 {kit.ci_str(la, ha)} / 候補 {kit.ci_str(lb, hb)}")

## 6. 塊の標準誤差
同じ種類の難問（8 問ずつ 6 種類）は一緒に崩れます。素朴な SE は小さすぎます（Miller 2024）。

In [ ]:
show(stats.cluster_se)
scen = [base[i]["scenario"] for i in ids]
naive, clus = stats.naive_se(b), stats.cluster_se(b, scen)
print(f"\n候補の正答（{len(set(scen))} 種類の塊）: 素朴な SE {naive:.3f} / 塊の SE {clus:.3f}（{clus / naive:.2f} 倍）")
ss = sorted(set(scen))
for part in (ss[:3], ss[3:]):
    print("   " + " / ".join(f"{s} {sum(b[j] for j in range(len(b)) if scen[j] == s)}/8" for s in part))

## 7. 検出力 ― 小さな差を見つけるには何問要るか（模擬）
本当の差が 3 ポイントあるとき、試験を 200 回やって「差あり」と言えた回数を数えます。

In [ ]:
rng = random.Random(0)
pd, delta = 0.29, 0.03            # 食い違う問いの割合と、本当の差 3 ポイント

def found(n):                     # 1 回の比較で「差あり」と言えたか（正規近似の McNemar）
    d = sum(rng.random() < pd for _ in range(n))
    c = sum(rng.random() < (pd + delta) / (2 * pd) for _ in range(d))   # 候補だけ正解
    return d > 0 and abs(c - (d - c)) / d ** 0.5 > 1.96

for n in (48, 500, 2500):
    hits = sum(found(n) for _ in range(200))
    print(f"{n:5d} 問の試験を 200 回: 差 3 ポイントを見つけた {hits:3d} 回（{hits / 2:.0f}%）")

## 8. 必要な件数を計算する
対あり・α=0.05 両側・検出力 80%。食い違う割合は 48 問の実測（14/48）を使います。

In [ ]:
show(stats.n_paired, 2, 6)
print(" 123      return ceil((za·√p_disc + zb·√(p_disc − delta²))² / delta²)   ← 長い行を縮めて表示")
pd = (only_a + only_b) / len(ids)
rows = [[f"{int(d * 100)} ポイント", stats.n_paired(d, max(pd, d * 1.5))] for d in (0.03, 0.05, 0.10, 0.20)]
print(f"\n食い違う問いの割合 {pd:.2f} と仮定")
print(kit.table(rows, ["見つけたい差", "必要な問いの数"]))

## 9. 多重比較の補正（Holm）
難問の種類ごとに 6 回検定すると、どれかが偶然「有意」になりやすい。

In [ ]:
show(stats.holm)
pv = {}
for s in ss:
    j = [i for i in range(len(b)) if scen[i] == s]
    pv[s] = stats.mcnemar_exact(*stats.discordant([a[i] for i in j], [b[i] for i in j]))
keep = stats.holm(pv)
for s, p in sorted(pv.items(), key=lambda kv: kv[1]):
    print(f"   {s:14s} p={p:.3f}  補正前 {'有意' if p < 0.05 else '-'} / Holm {'有意' if keep[s] else '-'}")

## 10. 方法で結論が割れるとき
短くした依頼文の下で、Sonnet と Opus を同じ 48 問で比べます（lab09 の本物の実行の記録）。

In [ ]:
opus = {r["id"]: r["correct"] for r in kit.read_jsonl("lab09_ci_gate/out/runs_candidate_v2_opus-5-5.jsonl")}
x, y = b, [opus[i] for i in ids]
s_only, o_only = stats.discordant(x, y)
bs = stats.paired_bootstrap(x, y, seed=0)
print(f"短い依頼文で Sonnet {sum(x)}/48・Opus {sum(y)}/48（同じ 48 問）")
print(f"Sonnet だけ正解 {s_only} / Opus だけ正解 {o_only}")
print(f"McNemar（正確）p={stats.mcnemar_exact(s_only, o_only):.3f}  → 5% では差があるとは言えない")
print(f"対のブートストラップ 95%区間 [{bs['lo']:+.3f}, {bs['hi']:+.3f}]  → 0 を含まない")
print("件数が少ないと、方法で結論が割れる。方法は先に 1 つ決め、割れたら件数を増やす")

## 11. 例題 ― 食い違い 14 対 0 を手で検定する
食い違いが本当に五分五分なら、14 問とも同じ側に寄る確率はどれだけか。

In [ ]:
n = 14 + 0                         # 基準だけ正解 14 問 / 候補だけ正解 0 問
one_side = 0.5 ** n                # 14 問とも基準の側に寄る確率
print(f"食い違い {n} 問が全部片側に寄る確率 = 0.5^{n} = {one_side:.6f}")
print(f"両側の p 値 = 2 × {one_side:.6f} = {2 * one_side:.6f}")
print(f"stats.mcnemar_exact(14, 0)     = {stats.mcnemar_exact(14, 0):.6f}")
print("→ p≈0.0001。偶然で 14 対 0 になることは、まず無い")

## 12. 例題 ― 3 ポイントの差に要る問いの数
`stats.n_paired` の式（Connor 1987 の近似）を一つずつ計算します。

In [ ]:
import math
d, pd = 0.03, 14 / 48              # 見つけたい差 3 ポイント、食い違う割合（実測）
za, zb = 1.96, 0.8416              # 両側 5% と、検出力 80%
r1, r2 = math.sqrt(pd), math.sqrt(pd - d * d)
num = (za * r1 + zb * r2) ** 2
print(f"√pd = {r1:.4f}、√(pd − d²) = {r2:.4f}")
print(f"分子 = ({za} × {r1:.4f} + {zb} × {r2:.4f})² = {num:.4f}")
print(f"必要な問い = 分子 ÷ d² = {num:.4f} ÷ {d * d:.4f} = {num / (d * d):.0f}")
print(f"stats.n_paired(0.03, {pd:.4f}) = {stats.n_paired(d, pd)}")
print("→ 約 2,500 問。いまの 48 問では、3 ポイントの差は見えない")